In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix


In [ ]:
# Charger le DataFrame
preddf = pd.read_csv("expe_log/preds.csv")

# Nettoyer les espaces dans les noms de colonnes
preddf.columns = preddf.columns.str.strip()

y_pred = preddf["preds"]
y_true = preddf["labels"]
y_pred = y_pred.map({"sain": 0, "malade": 1})
y_true = y_true.map({"sain": 0, "malade": 1})

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

def plot_confusion_barplots(y_true, y_pred, df, age_column, age_thresholds, labels,ymax=1):
    """
    Affiche des barplots organisés en grille (TN/FP en haut, FN/TP en bas) pour différents seuils d'âge avec Plotly.
    Ajoute des légendes avec "Vérité" sur l'axe Y et "Prédiction" sur l'axe X.
    Sépare en deux figures : une pour les âges > seuil et l'autre pour les âges < seuil.
    Normalisation globale : pour chaque seuil, la somme des 4 valeurs (TN, FP, FN, TP) est égale à 1.
    """
    results_greater = []
    results_lesser = []
    
    for threshold in age_thresholds:
        df[f'Age>{threshold}'] = df[age_column] > threshold
        group_greater = df[f'Age>{threshold}']
        group_lesser = ~group_greater
        y_true_group_greater = y_true[group_greater]
        y_pred_group_greater = y_pred[group_greater]
        y_true_group_lesser = y_true[group_lesser]
        y_pred_group_lesser = y_pred[group_lesser]
        
        cm_greater = confusion_matrix(y_true_group_greater, y_pred_group_greater, labels=[0, 1])
        cm_lesser = confusion_matrix(y_true_group_lesser, y_pred_group_lesser, labels=[0, 1])
        
        tn_g, fp_g, fn_g, tp_g = cm_greater.ravel()
        tn_l, fp_l, fn_l, tp_l = cm_lesser.ravel()
        
        total_greater = tn_g + fp_g + fn_g + tp_g
        total_lesser = tn_l + fp_l + fn_l + tp_l
        
        if total_greater > 0:
            tn_g, fp_g, fn_g, tp_g = tn_g / total_greater, fp_g / total_greater, fn_g / total_greater, tp_g / total_greater
        if total_lesser > 0:
            tn_l, fp_l, fn_l, tp_l = tn_l / total_lesser, fp_l / total_lesser, fn_l / total_lesser, tp_l / total_lesser
        
        results_greater.extend([
            {'Seuil': threshold, 'Catégorie': 'TN', 'Nombre': tn_g},
            {'Seuil': threshold, 'Catégorie': 'FP', 'Nombre': fp_g},
            {'Seuil': threshold, 'Catégorie': 'FN', 'Nombre': fn_g},
            {'Seuil': threshold, 'Catégorie': 'TP', 'Nombre': tp_g}
        ])
        
        results_lesser.extend([
            {'Seuil': threshold, 'Catégorie': 'TN', 'Nombre': tn_l},
            {'Seuil': threshold, 'Catégorie': 'FP', 'Nombre': fp_l},
            {'Seuil': threshold, 'Catégorie': 'FN', 'Nombre': fn_l},
            {'Seuil': threshold, 'Catégorie': 'TP', 'Nombre': tp_l}
        ])
    
    # Convertir les résultats en DataFrame
    results_greater_df = pd.DataFrame(results_greater)
    results_lesser_df = pd.DataFrame(results_lesser)
    
    # Créer la première figure pour les âges > seuil
    fig_greater = make_subplots(rows=2, cols=2, subplot_titles=["Vérité: sain / Prédiction: sain", "Vérité: sain / Prédiction: malade", "Vérité: malade / Prédiction: sain", "Vérité: malade / Prédiction: malade"])
    categories = ["TN", "FP", "FN", "TP"]
    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
    
    for category, (row, col) in zip(categories, positions):
        subset = results_greater_df[results_greater_df['Catégorie'] == category]
        fig_greater.add_trace(go.Bar(x=subset['Seuil'], y=subset['Nombre'], name=category), row=row, col=col)
    
    fig_greater.update_layout(
        title_text="Matrice de confusion pour les âges > seuil",
        height=600, width=800,
        xaxis_title="Prédiction",
        yaxis_title="Vérité"
    )
    
    # Créer la deuxième figure pour les âges < seuil
    fig_lesser = make_subplots(rows=2, cols=2, subplot_titles=["Vérité: sain / Prédiction: sain", "Vérité: sain / Prédiction: malade", "Vérité: malade / Prédiction: sain", "Vérité: malade / Prédiction: malade"])
    
    for category, (row, col) in zip(categories, positions):
        subset = results_lesser_df[results_lesser_df['Catégorie'] == category]
        fig_lesser.add_trace(go.Bar(x=subset['Seuil'], y=subset['Nombre'], name=category), row=row, col=col)
    
    fig_lesser.update_layout(
        title_text="Matrice de confusion pour les âges < seuil",
        height=600, width=800,
        xaxis_title="Prédiction",
        yaxis_title="Vérité"
    )

    # fig_greater.update_yaxes(range=[0, ymax])
    # fig_lesser.update_yaxes(range=[0, ymax])
    
    # Affichage des graphiques
    fig_greater.show()
    fig_lesser.show()

In [ ]:
age_thresholds = range(20, 80, 5)  # Seuils de 30 à 45 ans
plot_confusion_barplots(y_true, y_pred, preddf.copy(), "Patient Age", age_thresholds, labels=["sain", "malade"], ymax=0.7)


In [ ]:
# Charger le DataFrame
preddf = pd.read_csv("expe_log/preds.csv")

# Nettoyer les espaces dans les noms de colonnes
preddf.columns = preddf.columns.str.strip()

y_pred = preddf["preds"]
y_true = preddf["labels"]
y_pred = y_pred.map({"sain": 0, "malade": 1})
y_true = y_true.map({"sain": 0, "malade": 1})

In [ ]:
preddf['+40ans']=preddf['Patient Age'] >= 40

preddf

In [ ]:
import pandas as pd
from aif360.datasets import BinaryLabelDataset

# Charger le DataFrame
preddf = pd.read_csv("expe_log/preds.csv")

# Nettoyer les espaces dans les noms de colonnes
preddf.columns = preddf.columns.str.strip()

# Convertir les labels et les prédictions en numériques
preddf["preds"] = preddf["preds"].map({"sain": 0, "malade": 1})
preddf["labels"] = preddf["labels"].map({"sain": 0, "malade": 1})

# Convertir "Patient Gender" en numérique
preddf["Patient Gender"] = preddf["Patient Gender"].map({"M": 0, "F": 1})

# Définir l'attribut protégé (supposons qu'il représente l'âge > 40 ans)
if "+40ans" not in preddf.columns:
    preddf["+40ans"] = (preddf["Patient Age"] > 40).astype(int)  # Crée la colonne si nécessaire

protected_attribute = "+40ans"

label = 'Finding Labels'

# one hotage
df_ohe = preddf[label].str.get_dummies(sep='|').astype(bool)
# Join
preddf = preddf.drop(columns=['Finding Labels']).join(df_ohe)
print(preddf.columns)

# Supprimer les colonnes non pertinentes
filtered_df = preddf.drop(["View Position", "Finding Labels", "Image Index"], axis=1, errors="ignore")

# Vérifier que l'attribut protégé est bien dans les données
if protected_attribute in filtered_df.columns:
    dataset = BinaryLabelDataset(
        favorable_label=0,  # "Sain" est la classe favorable
        unfavorable_label=1,  # "Malade" est la classe défavorable
        df=filtered_df,
        label_names=["labels"],
        protected_attribute_names=[protected_attribute]
    )
else:
    print(f"Colonne protégée '{protected_attribute}' non trouvée dans le dataset.")


In [ ]:
# from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric

# # Créer l'objet de métriques pour le dataset original
# metric_orig = BinaryLabelDatasetMetric(dataset, 
#                                        privileged_groups=[{protected_attribute: 1}], 
#                                        unprivileged_groups=[{protected_attribute: 0}])

# print("📊 **Métriques AVANT correction**")
# print("Disparate Impact:", metric_orig.disparate_impact())
# print("Statistical Parity Difference:", metric_orig.statistical_parity_difference())

# # Si tu as les prédictions du modèle initial
# metric_classif_orig = ClassificationMetric(dataset, dataset, 
#                                            privileged_groups=[{protected_attribute: 1}], 
#                                            unprivileged_groups=[{protected_attribute: 0}])
# print("Equal Opportunity Difference:", metric_classif_orig.equal_opportunity_difference())
# print("Average Odds Difference:", metric_classif_orig.average_odds_difference())


In [ ]:
# from aif360.algorithms.preprocessing import Reweighing

# # Appliquer le reweighing
# rw = Reweighing(privileged_groups=[{protected_attribute: 1}], 
#                 unprivileged_groups=[{protected_attribute: 0}])
# dataset_transf = rw.fit_transform(dataset)

# print("Poids moyens après reweighing :", dataset_transf.instance_weights.mean())


In [ ]:
# # Calcul des nouvelles métriques après correction
# metric_transf = BinaryLabelDatasetMetric(dataset_transf, 
#                                          privileged_groups=[{protected_attribute: 1}], 
#                                          unprivileged_groups=[{protected_attribute: 0}])

# print("\n📊 **Métriques APRÈS correction**")
# print("Disparate Impact:", metric_transf.disparate_impact())
# print("Statistical Parity Difference:", metric_transf.statistical_parity_difference())

# # Si tu as de nouvelles prédictions
# metric_classif_transf = ClassificationMetric(dataset_transf, dataset_transf, 
#                                              privileged_groups=[{protected_attribute: 1}], 
#                                              unprivileged_groups=[{protected_attribute: 0}])
# print("Equal Opportunity Difference:", metric_classif_transf.equal_opportunity_difference())
# print("Average Odds Difference:", metric_classif_transf.average_odds_difference())


In [ ]:
from aif360.sklearn.metrics import *


def get_group_metrics(
    y_true,
    y_pred=None,
    prot_attr=None,
    priv_group=1,
    pos_label=1,
    sample_weight=None,
):
    group_metrics = {}
    group_metrics["base_rate"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
    return group_metrics

In [ ]:
import pandas as pd
from aif360.datasets import BinaryLabelDataset

# Charger le DataFrame
preddf = pd.read_csv("expe_log/preds.csv")

# Nettoyer les espaces dans les noms de colonnes
preddf.columns = preddf.columns.str.strip()

# Convertir les labels et les prédictions en numériques
preddf["preds"] = preddf["preds"].map({"sain": 0, "malade": 1})
preddf["labels"] = preddf["labels"].map({"sain": 0, "malade": 1})

# Convertir "Patient Gender" en numérique
preddf["Patient Gender"] = preddf["Patient Gender"].map({"M": 0, "F": 1})

# Définir l'attribut protégé (supposons qu'il représente l'âge > 40 ans)
if "+40ans" not in preddf.columns:
    preddf["+40ans"] = (preddf["Patient Age"] > 40).astype(int)  # Crée la colonne si nécessaire

protected_attribute = "+40ans"

label = 'Finding Labels'

# one hotage
df_ohe = preddf[label].str.get_dummies(sep='|').astype(bool)
# Join
preddf = preddf.drop(columns=['Finding Labels']).join(df_ohe)
print(preddf.columns)

# Supprimer les colonnes non pertinentes
filtered_df = preddf.drop(["View Position", "Finding Labels", "Image Index"], axis=1, errors="ignore")

# Vérifier que l'attribut protégé est bien dans les données
if protected_attribute in filtered_df.columns:
    dataset = BinaryLabelDataset(
        favorable_label=0,  # "Sain" est la classe favorable
        unfavorable_label=1,  # "Malade" est la classe défavorable
        df=filtered_df,
        label_names=["labels"],
        protected_attribute_names=[protected_attribute]
    )
else:
    print(f"Colonne protégée '{protected_attribute}' non trouvée dans le dataset.")


metrics = get_group_metrics(
    y_true=preddf["labels"],
    y_pred=preddf["preds"],
    prot_attr=preddf["+40ans"],
    priv_group=1,
    pos_label=1
)

# Affichage des résultats
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Initialiser l'algorithme de Reweighing
rw = Reweighing(unprivileged_groups=[{"+40ans": 0}], privileged_groups=[{"+40ans": 1}])

# Appliquer Reweighing sur le dataset
rw.fit(dataset)
transformed_dataset = rw.transform(dataset)

preddf["WEIGHTS"] = transformed_dataset.instance_weights

# Vérification des poids ajustés
print("Poids ajustés des instances :", transformed_dataset.instance_weights.mean())  # Afficher les premiers poids

# Maintenan

In [ ]:
from sklearn.model_selection import train_test_split
from train_classifieur import train_classifier, pred_classifier

# Diviser le dataset en données d'entraînement et de test
X = transformed_dataset.features
y = transformed_dataset.labels
sample_weight = transformed_dataset.instance_weights

# Diviser les données d'entraînement et de test
X_train, X_test, y_train, y_test, sample_weight_train, sample_weight_test = train_test_split(X, y, sample_weight, test_size=0.2, random_state=42)

# Entraîner le modèle avec les poids ajustés
ckpt_path, ckpt_score = train_classifier(
    logdir="./expe_log/",
    datadir="./data/SAILLANT_ARTHUR/selected_data/",
    csv="./data/SAILLANT_ARTHUR/selected_data/metadata.csv",
)

# Faire des prédictions
pred_classifier(
    datadir="./data/SAILLANT_ARTHUR/selected_data/",
    csv_in="./data/SAILLANT_ARTHUR/selected_data/metadata.csv",
    csv_out="./expe_log/preds.csv",
    ckpt_path=ckpt_path
)



In [ ]:
# Charger les prédictions pour l'évaluation
preddf = pd.read_csv("./expe_log/preds.csv")

# Convertir les labels et les prédictions en numériques
preddf["preds"] = preddf["preds"].map({"sain": 0, "malade": 1})
preddf["labels"] = preddf["labels"].map({"sain": 0, "malade": 1})

# Convertir "Patient Gender" en numérique
preddf["Patient Gender"] = preddf["Patient Gender"].map({"M": 0, "F": 1})

# Définir l'attribut protégé (supposons qu'il représente l'âge > 40 ans)
if "+40ans" not in preddf.columns:
    preddf["+40ans"] = (preddf["Patient Age"] > 40).astype(int)  # Crée la colonne si nécessaire


metrics = get_group_metrics(
    y_true=preddf["labels"],
    y_pred=preddf["preds"],
    prot_attr=preddf["+40ans"],
    priv_group=1,
    pos_label=1
)

# Affichage des résultats de fairness
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Comparaison des changements dans les métriques
print("\nComparaison des changements dans les métriques :")
for metric in metrics_before.keys():
    before = metrics_before[metric]
    after = metrics_after[metric]
    change = after - before
    print(f"{metric}: Avant = {before:.4f}, Après = {after:.4f}, Changement = {change:.4f}")